In [1]:
import pandas as pd
import numpy as np

In [2]:
filename = 'diabetes_dataset.csv'
df = pd.read_csv(filename)
df.head(1)

,age,gender,ethnicity,education_level,income_level,employment_status,smoking_status,alcohol_consumption_per_week,physical_activity_minutes_per_week,diet_score,...,hdl_cholesterol,ldl_cholesterol,triglycerides,glucose_fasting,glucose_postprandial,insulin_level,hba1c,diabetes_risk_score,diabetes_stage,diagnosed_diabetes
0,58,Male,Asian,Highschool,Lower-Middle,Employed,Never,0,215,5.7,...,41,160,145,136,236,6.36,8.18,29.6,Type 2,1


In [3]:
df['systolic_diastolic_ratio'] = df['systolic_bp'] / df['diastolic_bp']
df['hdl_ldl_ratio'] = df['hdl_cholesterol'] / df['ldl_cholesterol']
df['glucose_after_eating_to_fasting_ratio'] = df['glucose_postprandial'] / df['glucose_fasting']
df ['bmi_to_age_ratio'] = df['bmi'] / df['age']
df['bmi_cholesterol_ratio'] = df['bmi'] / df['cholesterol_total']

In [4]:
input_cols = ['age', 'gender','smoking_status', 'alcohol_consumption_per_week',
       'sleep_hours_per_day' , 'family_history_diabetes', 'hypertension_history',
       'cardiovascular_history', 'bmi','cholesterol_total' , 'triglycerides', 'insulin_level', 'hba1c', 
       'physical_activity_minutes_per_week', 'diet_score',
       'systolic_diastolic_ratio', 'hdl_ldl_ratio',
       'glucose_after_eating_to_fasting_ratio', 'bmi_to_age_ratio',
       'bmi_cholesterol_ratio']

target_col = 'diabetes_stage'

In [5]:
numerical_cols = df[input_cols].select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = df[input_cols].select_dtypes(include=['object']).columns.tolist()

In [6]:
from sklearn.model_selection import train_test_split
X = df[input_cols]
y = df[target_col]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)
X_train.shape, X_test.shape

((90000, 20), (10000, 20))

In [7]:
from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
ohe.fit(X_train[cat_cols])
ohe.categories_

[array(['Female', 'Male', 'Other'], dtype=object),
 array(['Current', 'Former', 'Never'], dtype=object)]

In [8]:
X_train_ohe = ohe.transform(X_train[cat_cols])
X_test_ohe = ohe.transform(X_test[cat_cols])

In [9]:
input_cols_final = numerical_cols + ohe.get_feature_names_out(cat_cols).tolist()

In [10]:
X_train_final = pd.DataFrame(X_train_ohe, columns=ohe.get_feature_names_out(cat_cols), index=X_train.index)
X_train_final[numerical_cols] = X_train[numerical_cols]
X_test_final = pd.DataFrame(X_test_ohe, columns=ohe.get_feature_names_out(cat_cols), index=X_test.index)
X_test_final[numerical_cols] = X_test[numerical_cols]
X_train_final.head()

,gender_Female,gender_Male,gender_Other,smoking_status_Current,smoking_status_Former,smoking_status_Never,age,alcohol_consumption_per_week,sleep_hours_per_day,family_history_diabetes,...,triglycerides,insulin_level,hba1c,physical_activity_minutes_per_week,diet_score,systolic_diastolic_ratio,hdl_ldl_ratio,glucose_after_eating_to_fasting_ratio,bmi_to_age_ratio,bmi_cholesterol_ratio
51994,1.0,0.0,0.0,0.0,0.0,1.0,67,2,8.9,0,...,93,14.90,5.86,34,5.0,2.015385,0.405405,1.731183,0.434328,0.125431
77540,1.0,0.0,0.0,0.0,0.0,1.0,39,1,7.1,0,...,129,3.49,6.82,95,8.6,1.739130,0.420168,1.542857,0.607692,0.119697
16382,1.0,0.0,0.0,1.0,0.0,0.0,36,5,6.1,0,...,167,15.47,7.10,419,5.0,1.585714,0.650000,1.596491,0.691667,0.141477
83439,1.0,0.0,0.0,0.0,0.0,1.0,62,1,7.2,0,...,112,13.97,7.50,55,2.6,1.686047,0.388350,1.458015,0.574194,0.189362
61618,1.0,0.0,0.0,1.0,0.0,0.0,31,3,6.8,0,...,98,7.59,7.40,138,6.1,1.541667,0.343511,1.660714,1.054839,0.151389


In [11]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_train_enc = le.fit_transform(y_train)
y_test_enc  = le.transform(y_test)

In [12]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, StackingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression

In [18]:
base_learners = [
    ('rf', RandomForestClassifier(n_estimators=200, max_depth=8, max_features=14,random_state=42)),
    ('xgb', XGBClassifier(n_estimators=200,eval_metric='logloss', random_state=42)),
    ('lgbm', LGBMClassifier(n_estimators=200, max_depth=9,random_state=42))
]
# parameters are tuned according to previous models

In [19]:
# Meta model
meta_learner = LogisticRegression()

In [23]:
# Stacking model
stack_model = StackingClassifier(
    estimators=base_learners,
    final_estimator=meta_learner,
    cv=5,
    stack_method='predict_proba',   # use probabilities instead of hard predictions
    passthrough=False,
    n_jobs=-1
)

In [24]:
stack_model.fit(X_train_final, y_train_enc)

,estimators,"[('rf', ...), ('xgb', ...), ...]"
,final_estimator,LogisticRegression()
,cv,5
,stack_method,'predict_proba'
,n_jobs,-1
,passthrough,False
,verbose,0
,n_estimators,200
,criterion,'gini'
,max_depth,8
,min_samples_split,2


In [25]:
stack_model.score(X_test_final, y_test_enc)

0.8742